# Unidad 3 - Sistema de Recomendación con TensorFlow Recommenders

In [1]:
!pip install -q tensorflow tensorflow-recommenders tensorflow-datasets scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 7.7 MB/s eta 0:00:00


In [3]:
# Importar bibliotecas
import tensorflow as tf
import tensorflow_recommenders as tfrs
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, roc_auc_score
import os, random
from sklearn.metrics import roc_auc_score

## Importar librerías, comprobar GPU y cargar MovieLens

In [ ]:
# Fijar semillas para reproducibilidad (aproximada en GPU)
def fijar_semillas(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
fijar_semillas(42)

# Comprobar GPU y activar "memory growth" (opcional recomendado)
gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow version:", tf.__version__)
print("GPUs detectadas:", gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Memoria de GPU en modo 'growth' activada.")
    except Exception as e:
        print("Aviso al configurar memoria de GPU:", e)

# Cargar dataset MovieLens 100k (ratings)
ratings_ds = tfds.load("movielens/100k-ratings", split="train", as_supervised=False)
movies_ds  = tfds.load("movielens/100k-movies",  split="train", as_supervised=False)

# Mostrar ejemplo de estructura de un registro de rating
print("\nEjemplo de 3 ratings:")
for i, x in enumerate(ratings_ds.take(3)):
    # Campos comunes: 'user_id', 'movie_title', 'user_rating', 'timestamp'
    uid = x.get("user_id").numpy().decode("utf-8")
    title = x.get("movie_title").numpy().decode("utf-8")
    rating = float(x.get("user_rating").numpy())
    print(f"- {i+1}) user_id={uid} | movie_title='{title}' | rating={rating}")

# Extraer vocabularios únicos (para embeddings más adelante)
def recolectar_vocab(ds, clave, limite=5000):
    # Recoger valores hasta un límite para evitar consumo de memoria alto
    vals = set()
    for x in ds.take(limite):
        vals.add(x.get(clave).numpy().decode("utf-8"))
    return sorted(list(vals))

print("\nRecolectar vocabularios preliminares...")
usuarios_preview = recolectar_vocab(ratings_ds, "user_id", limite=10000)
peliculas_preview = recolectar_vocab(movies_ds,  "movie_title", limite=10000)

print(f"Usuarios únicos (muestra): {len(usuarios_preview)}")
print(f"Películas únicas (muestra): {len(peliculas_preview)}")

# Guardar estos vocabularios preliminares en memoria para uso posterior
usuario_vocab = usuarios_preview
pelicula_vocab = peliculas_preview

TensorFlow version: 2.19.0
GPUs detectadas: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Memoria de GPU en modo 'growth' activada.


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

...

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/movielens/100k-movies/incomplete.G8F1CB_0.1.1/movielens-train.tfrecord*...…

Dataset movielens downloaded and prepared to /root/tensorflow_datasets/movielens/100k-movies/0.1.1. Subsequent calls will reuse this data.

Ejemplo de 3 ratings:
- 1) user_id=138 | movie_title='One Flew Over the Cuckoo's Nest (1975)' | rating=4.0
- 2) user_id=92 | movie_title='Strictly Ballroom (1992)' | rating=2.0
- 3) user_id=301 | movie_title='Very Brady Sequel, A (1996)' | rating=4.0

Recolectar vocabularios preliminares...
Usuarios únicos (muestra): 923
Películas únicas (muestra): 1664


## Preparar splits (train/val) y pipelines tf.data

In [11]:
# 0) Obtener nombres de géneros desde el builder (ClassLabel)
movies_builder = tfds.builder("movielens/100k-movies")
movies_builder.download_and_prepare()  # no vuelve a bajar si ya está
genre_names = movies_builder.info.features["movie_genres"].feature.names  # lista de strings
# Asegurar "Unknown" en el vocabulario
if "Unknown" not in genre_names:
    genre_names = ["Unknown"] + genre_names

# 1) Recoger (título, género_principal como string) desde movies_ds
def obtener_titulo_y_genero_principal(movies_iter):
    titulos, generos_principales = [], []
    for x in movies_iter:
        t = x["movie_title"].numpy().decode("utf-8")
        gvals = x["movie_genres"].numpy()  # puede ser array de ints
        if gvals.size == 0:
            gp = "Unknown"
        else:
            gp_idx = int(gvals[0])         # primer género como "principal"
            # proteger rango
            if 0 <= gp_idx < len(genre_names):
                gp = genre_names[gp_idx]
            else:
                gp = "Unknown"
        titulos.append(t)
        generos_principales.append(gp)
    return titulos, generos_principales

movies_list = list(movies_ds)  # seguro: ~1682 items
titles_py, primary_genres_py = obtener_titulo_y_genero_principal(movies_list)

# 2) Vocabulario de géneros y StringLookup para género
generos_unicos = sorted(list(set(primary_genres_py + ["Unknown"])))
genre_lookup = tf.keras.layers.StringLookup(vocabulary=generos_unicos, mask_token=None)
num_genres = genre_lookup.vocabulary_size()
print(f"Vocab géneros: {num_genres} -> {generos_unicos[:10]}...")

# 3) Tabla título -> genre_idx (int64)
titles_tensor = tf.constant(titles_py)
genres_idx_tensor = genre_lookup(tf.constant(primary_genres_py))  # int64
initializer = tf.lookup.KeyValueTensorInitializer(
    keys=titles_tensor, values=tf.cast(genres_idx_tensor, tf.int64)
)
title2genre_table = tf.lookup.StaticHashTable(initializer, default_value=tf.constant(0, tf.int64))  # 0=Unknown

# 4) Extender proc_example con genre_idx
@tf.function
def proc_example(x):
    uid = user_lookup(x["user_id"])
    mid = movie_lookup(x["movie_title"])
    gid = title2genre_table.lookup(x["movie_title"])  # int64
    return {"user_id": uid, "movie_title": mid, "genre_idx": tf.cast(gid, tf.int32)}

# 5) Pipelines finales (ajustar batch_size si hace falta)
batch_size = 4096
train_proc = (train_ds
              .map(proc_example, num_parallel_calls=tf.data.AUTOTUNE)
              .cache()
              .batch(batch_size)
              .prefetch(tf.data.AUTOTUNE))

val_proc = (val_ds
            .map(proc_example, num_parallel_calls=tf.data.AUTOTUNE)
            .cache()
            .batch(batch_size)
            .prefetch(tf.data.AUTOTUNE))

print("Pipelines con 'genre_idx' preparados (Colab).")


Vocab géneros: 20 -> ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir']...
Pipelines con 'genre_idx' preparados (Colab).


## Modelo TFRS Two-Tower + Índice manual

In [ ]:
# 1) Hiperparámetros
embedding_dim_id   = 128   # embedding para movie_id
embedding_dim_gen  = 32    # embedding para género (más pequeño)
proj_dim           = 128   # dimensión final de proyección
learning_rate      = 5e-4
epochs             = 20
l2_reg             = 1e-6
drop_rate          = 0.15

# 2) Capas de embedding
user_embedding  = tf.keras.layers.Embedding(
    input_dim=num_users,  output_dim=proj_dim,
    embeddings_regularizer=tf.keras.regularizers.l2(l2_reg)
)
movieid_embedding = tf.keras.layers.Embedding(
    input_dim=num_movies, output_dim=embedding_dim_id,
    embeddings_regularizer=tf.keras.regularizers.l2(l2_reg)
)
genre_embedding = tf.keras.layers.Embedding(
    input_dim=num_genres, output_dim=embedding_dim_gen,
    embeddings_regularizer=tf.keras.regularizers.l2(l2_reg)
)

# 3) Modelos "raw" desde strings (para índice manual)
class UserModelFromString(tf.keras.Model):
    def __init__(self, lookup_user, embed_user):
        super().__init__()
        self.lookup_user = lookup_user
        self.embed_user  = embed_user
    def call(self, x):
        uid = self.lookup_user(x)               # string -> int
        return self.embed_user(uid)             # [B, proj_dim]

class MovieModelFromString(tf.keras.Model):
    def __init__(self, lookup_title, embed_movieid, embed_genre, table_title2genre):
        super().__init__()
        self.lookup_title = lookup_title
        self.embed_movieid= embed_movieid
        self.embed_genre  = embed_genre
        self.table_t2g    = table_title2genre   # tf.lookup.StaticHashTable

        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(drop_rate),
            tf.keras.layers.Dense(proj_dim)
        ])

    def call(self, x_title):
        # x_title: tensor de strings con el título
        mid = self.lookup_title(x_title)                        # [B]
        gid = self.table_t2g.lookup(x_title)                    # [B], int64
        gid = tf.where(gid < 0, tf.zeros_like(gid), gid)        # fallback a 0 ("Unknown")
        z_id  = self.embed_movieid(mid)                         # [B, embedding_dim_id]
        z_gen = self.embed_genre(tf.cast(gid, tf.int32))        # [B, embedding_dim_gen]
        z     = tf.concat([z_id, z_gen], axis=-1)               # [B, embedding_dim_id+embedding_dim_gen]
        return self.mlp(z)                                      # [B, proj_dim]

user_model_raw  = UserModelFromString(user_lookup, user_embedding)
movie_model_raw = MovieModelFromString(movie_lookup, movieid_embedding, genre_embedding, title2genre_table)

# 4) Modelo Two-Tower (entrena con IDs enteros ya mapeados en proc_example)
class RetrievalTwoTower(tfrs.models.Model):
    def __init__(self, user_embed, movieid_embed, genre_embed):
        super().__init__()
        self.user_tower = tf.keras.Sequential([
            user_embed,
            # ya queda en proj_dim
        ])
        # Torre de item: concat(emb_id, emb_genre) -> MLP -> proj_dim
        self.movieid_embed = movieid_embed
        self.genre_embed   = genre_embed
        self.item_mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dropout(drop_rate),
            tf.keras.layers.Dense(proj_dim)
        ])
        self.task = tfrs.tasks.Retrieval(metrics=None)

    def compute_loss(self, features, training=False):
        # features: {'user_id': int, 'movie_title': int, 'genre_idx': int}
        u = self.user_tower(features["user_id"])                           # [B, proj_dim]
        z_id  = self.movieid_embed(features["movie_title"])                # [B, embedding_dim_id]
        z_gen = self.genre_embed(features["genre_idx"])                    # [B, embedding_dim_gen]
        v_in  = tf.concat([z_id, z_gen], axis=-1)
        v     = self.item_mlp(v_in)                                        # [B, proj_dim]
        return self.task(u, v)

model = RetrievalTwoTower(user_embedding, movieid_embedding, genre_embedding)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate))

history = model.fit(
    train_proc,
    validation_data=val_proc,
    epochs=epochs,
    verbose=1
)

# 5) Índice manual de candidatos (ahora usando título -> género dentro de movie_model_raw)
candidates_ds = movies_ds.map(lambda x: x["movie_title"], num_parallel_calls=tf.data.AUTOTUNE)

titles_list = []
emb_chunks = []
for batch_titles in candidates_ds.batch(512):
    emb = movie_model_raw(batch_titles)  # usa lookup de título y tabla para género
    emb_chunks.append(emb)
    titles_list.extend([t.numpy().decode("utf-8") for t in batch_titles])

item_matrix   = tf.concat(emb_chunks, axis=0)     # [N_items, proj_dim]
titles_tensor = tf.constant(titles_list)          # [N_items], string
print(f"\nÍndice manual (con género): {item_matrix.shape[0]} películas indexadas.")

# 6) Recomendación top-K
import numpy as np

def recomendar_topk(user_id_str: str, k: int = 5):
    q = user_model_raw(tf.constant([user_id_str]))                 # [1, proj_dim]
    scores = tf.linalg.matmul(q, item_matrix, transpose_b=True)[0].numpy()
    order = np.argsort(scores)[::-1]
    vistos = set()
    rec_titles, rec_scores = [], []
    for idx in order:
        t = titles_list[idx]
        if t not in vistos:
            vistos.add(t)
            rec_titles.append(t)
            rec_scores.append(float(scores[idx]))
            if len(rec_titles) == k:
                break
    return rec_scores, rec_titles

scores, titles = recomendar_topk("138", k=5)
print("\nRecomendaciones top-5 (con género) para user_id=138:")
for s, t in zip(scores, titles):
    print(f"- score={s:.4f} | título={t}")


Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 16s 478ms/step - loss: 32417.2852 - regularization_loss: 2.7913e-04 - total_loss: 32417.2852 - val_loss: 29625.9141 - val_regularization_loss: 2.7988e-04 - val_total_loss: 29625.9141
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 32408.2031 - regularization_loss: 2.8292e-04 - total_loss: 32408.2031 - val_loss: 29620.0742 - val_regularization_loss: 2.8680e-04 - val_total_loss: 29620.0742
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 32377.9512 - regularization_loss: 2.9406e-04 - total_loss: 32377.9512 - val_loss: 29576.3281 - val_regularization_loss: 3.0335e-04 - val_total_loss: 29576.3281
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32239.2949 - regularization_loss: 3.1939e-04 - total_loss: 32239.2949 - val_loss: 29404.7168 - val_regularization_loss: 3.3958e-04 - val_total_loss: 29404.7168
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 31891.2285 - regularization_loss: 3.6716e-04 - total_loss: 3189

## Precision@K y AUC

In [ ]:
# 1) Recoger interacciones por usuario (strings) en train y val
def recolectar_interacciones(ds):
    # ds: tf.data con {"user_id": string, "movie_title": string}
    d = {}
    for x in ds:
        u = x["user_id"].numpy().decode("utf-8")
        m = x["movie_title"].numpy().decode("utf-8")
        if u not in d:
            d[u] = set()
        d[u].add(m)
    return d

train_inter = recolectar_interacciones(train_ds)  # del Bloque 2 (strings)
val_inter   = recolectar_interacciones(val_ds)    # del Bloque 2 (strings)

n_users_val = len(val_inter)
print(f"Usuarios con interacciones en validación: {n_users_val}")

# 2) Mapa título -> índice en nuestro índice manual
title_to_idx = {t: i for i, t in enumerate(titles_list)}  # titles_list viene del Bloque 3

# 3) Función auxiliar: top-K excluyendo ítems vistos en train
def recomendar_topk_excluyendo_train(user_id_str: str, k: int = 10):
    # Obtener embedding de usuario
    q = user_model_raw(tf.constant([user_id_str]))  # [1, D]
    # Puntuar todos los ítems
    scores = tf.linalg.matmul(q, item_matrix, transpose_b=True)[0].numpy()  # [N_items]
    # Enmascarar ítems vistos en TRAIN
    vistos_train = train_inter.get(user_id_str, set())
    for m in vistos_train:
        idx = title_to_idx.get(m, None)
        if idx is not None:
            scores[idx] = -1e9  # poner puntaje muy bajo
    # Seleccionar top-K
    topk_idx = np.argpartition(scores, -k)[-k:]
    topk_idx = topk_idx[np.argsort(scores[topk_idx])[::-1]]  # ordenar descendente
    return topk_idx, scores

# 4) Calcular Precision@K por usuario y promedio
def precision_at_k_usuario(user_id_str: str, k: int = 10):
    # Relevantes: películas del usuario en VALIDACIÓN
    relevantes = val_inter.get(user_id_str, set())
    if not relevantes:
        return None  # sin datos
    # Top-K excluyendo train
    topk_idx, _ = recomendar_topk_excluyendo_train(user_id_str, k=k)
    topk_titulos = [titles_list[i] for i in topk_idx]
    aciertos = sum(1 for t in topk_titulos if t in relevantes)
    return aciertos / k

def precision_at_k_global(k: int = 10):
    vals = []
    for u in val_inter.keys():
        p = precision_at_k_usuario(u, k=k)
        if p is not None:
            vals.append(p)
    return float(np.mean(vals)), len(vals)

# 5) Calcular AUC por usuario (muestreo de negativos)
rng = np.random.default_rng(42)

def auc_usuario(user_id_str: str, num_neg=500):
    relevantes = val_inter.get(user_id_str, set())
    if not relevantes:
        return None
    # Top-K (sólo para obtener vector de scores completo y máscara train)
    _, scores = recomendar_topk_excluyendo_train(user_id_str, k=10)

    # Positivos: títulos en VALIDACIÓN que existan en el índice
    pos_idx = [title_to_idx[t] for t in relevantes if t in title_to_idx]
    if not pos_idx:
        return None

    # Negativos: muestrear ítems que no estén en train ni en val del usuario
    vistos_total = set.union(train_inter.get(user_id_str, set()), relevantes)
    candidatos_neg = [title_to_idx[t] for t in titles_list if t not in vistos_total]
    if len(candidatos_neg) == 0:
        return None
    if len(candidatos_neg) > num_neg:
        neg_idx = rng.choice(candidatos_neg, size=num_neg, replace=False)
    else:
        neg_idx = np.array(candidatos_neg, dtype=int)

    # Construir y_true e y_score
    y_true  = np.concatenate([np.ones(len(pos_idx)), np.zeros(len(neg_idx))]).astype(int)
    y_score = np.concatenate([scores[pos_idx],         scores[neg_idx]       ]).astype(float)

    try:
        return roc_auc_score(y_true, y_score)
    except ValueError:
        # Caso raro: todos positivos/negativos idénticos
        return None

def auc_global(num_neg=500):
    vals = []
    for u in val_inter.keys():
        a = auc_usuario(u, num_neg=num_neg)
        if a is not None:
            vals.append(a)
    return float(np.mean(vals)), len(vals)

# 6) Ejecutar evaluación
K = 10
p_at_k, n_p = precision_at_k_global(k=K)
auc_mean, n_auc = auc_global(num_neg=500)

print(f"- Precision@{K}: {p_at_k:.4f} (usuarios válidos: {n_p})")
print(f"- AUC (pairwise, muestreo 500 neg): {auc_mean:.4f} (usuarios válidos: {n_auc})")


Usuarios con interacciones en validación: 942

RESULTADOS OFFLINE (validación):
- Precision@10: 0.0126 (usuarios válidos: 942)
- AUC (pairwise, muestreo 500 neg): 0.4751 (usuarios válidos: 942)


## Conclusión

En este trabajo práctico implementamos un sistema de recomendación con TensorFlow Recommenders (TFRS) utilizando el conjunto de datos MovieLens 100k. El objetivo fue construir un modelo de tipo retrieval two-tower, donde una torre representa a los usuarios y otra a los ítems (películas), y después evaluar el desempeño con las métricas Precision@K y AUC.

El flujo general fue: cargar y preparar los datos con tf.data, definir las capas de embeddings, entrenar el modelo básico con los identificadores de usuario y de película, y construir un índice manual para obtener recomendaciones top-K. Más adelante intentamos mejorar la calidad incluyendo características adicionales como el género principal de cada película y ajustando hiperparámetros (dimensión de los embeddings, número de épocas y tamaño de lote).

Durante el proceso enfrentamos varias dificultades. Primero, incompatibilidades entre Keras 3 y TFRS obligaron a adaptar el código para evitar el uso directo de FactorizedTopK y BruteForce. También notamos que la métrica Precision@10 resultó ser muy baja y que el AUC se situó cercano a valores aleatorios. Esto refleja la limitación de usar únicamente identificadores como entrada: el modelo no dispone de suficiente información semántica para generalizar bien. Al probar con géneros de películas, los resultados apenas cambiaron, lo que muestra la necesidad de un preprocesamiento más cuidadoso (por ejemplo, filtrar solo ratings positivos o representar todos los géneros en lugar de uno solo).

En resumen, cumplimos los pasos pedidos en el enunciado: diseñamos, entrenamos y evaluamos un recomendador con TFRS. Sin embargo, los resultados obtenidos muestran que un modelo tan básico tiene un desempeño limitado. Esto es valioso para el aprendizaje, ya que nos permitió entender las debilidades del enfoque y proponer líneas de mejora claras: enriquecer las features (géneros múltiples, año de estreno, ratings filtrados), explorar arquitecturas más profundas y comparar con baselines de popularidad o modelos colaborativos clásicos.